# Federated Learning with Local Differential Privacy

Worst-case FL scenario: Local DP (per-client noise) amplifies privacy cost. Uses [PFL](https://apple.github.io/pfl-research/).

This notebook runs federated learning (FL) with local differential privacy (LDP) as worst-case scenario for accuracy drop in FL with DP.
We simulate FL settings using the private federated learning library ([PFL](https://apple.github.io/pfl-research/)).

In [ ]:

import os
import torch
import types
import pandas as pd
import json


import nest_asyncio
nest_asyncio.apply()

from pfl.internal.ops import pytorch_ops
from pfl.model.pytorch import PyTorchModel
from pfl.internal.ops.selector import set_framework_module
from pfl.aggregate.simulate import SimulatedBackend
from pfl.algorithm import FederatedAveraging, NNAlgorithmParams
from pfl.callback import CentralEvaluationCallback, AggregateMetricsToDisk
from pfl.hyperparam import NNTrainHyperParams, NNEvalHyperParams
from pfl.data.dataset import Dataset
from pfl.privacy import PLDPrivacyAccountant
from pfl.privacy import GaussianMechanism


device = "cuda" if torch.cuda.is_available() else "cpu"
# If on Apple Silicon, use MPS
# device = "mps"
# os.environ['PFL_PYTORCH_DEVICE'] = device


os.environ['PFL_PYTORCH_DEVICE'] = device
set_framework_module(pytorch_ops)



sys.path.append('./dataset/')
from dataset.fashion_mnist.load_preprocess import load_and_preprocess_fashion_mnist_federated
from dataset.mnist.load_preprocess import load_and_preprocess_mnist_federated

sys.path.append('./utils/')
from utils.models import ThreeLayerNN
from utils.metrics import image_classification_loss, image_classification_metrics
from utils.pfl_logging_parser import save_best_stats_per_iteration

## Model
The model used for the experiments is a 3 layers neural network. Here, the model is adapted to the federated settings via the `get_pfl_model` function.

In [ ]:
def get_pfl_model():
	model = ThreeLayerNN(
		input_size=784,
		hidden_size=100,
		output_size=10
	)


	model.loss = types.MethodType(image_classification_loss,
									model)
	model.metrics = types.MethodType(image_classification_metrics,
										model)
	print(f'PFL model: \n{model}')
	pfl_params = [p for p in model.parameters() if p.requires_grad]


	# Since we rely on FederatedAveraging, which averages the model updates from clients,
	# we set the central learning rate to 1.0 to avoid scaling the updates again.
	central_learning_rate = 1.0

	pfl_model = PyTorchModel(model=model,
				local_optimizer_create=torch.optim.SGD,
				central_optimizer=torch.optim.SGD(pfl_params, central_learning_rate))
	
	return pfl_model

## Dataset
The experiments can be run on either MNIST or Fashion-MNIST datasets by changing the `dataset` variable below.

In [ ]:
dataset_name = "mnist"
# dataset_name = "fashion_mnist"

## Training

### Hyperparameter setting

In [ ]:
learning_rate = 0.1
momentum = 0
weight_decay = 0
clip_bound = 0.3
local_iterations = 5 
central_iterations = 1000
mnist_sample_size = 60000
population_size = 1000
cohort_size = 100
samples_per_user = mnist_sample_size // population_size
epochs = 10
local_batch_size = 10

### DP parameters
Among the DP parameters, we define $\epsilon$, $\delta$ and the clipping bound to clip gradients. We leverage sub-sampling privacy amplification by sampling `cohort_size` users from the whole `population_size`.

In [ ]:
sampling_probability = cohort_size / population_size
eps = 1.0
delta = 1e-5


### Training loop
Here, we define the training loop which first converts the dataset into a federated variant using utilities from the `pfl` library (More details in `dataset/mnist/load_and_preprocess_mnist_federated.py` and `dataset/fashion_mnist/load_and_preprocess_fashion_mnist_federated.py`). Then, we define the federated learning network and the local code that each client and the central server have to execute. We selected as learning algorithm Federated Averaging (FedAvg).

#### Local DP 
For local DP, we use Gaussian noise and the accounting uses the Privacy Loss Distribution (PLD). 

In [ ]:
base_folder_metrics = "local_dp_fl_training"
base_path_metrics = f"{base_folder_metrics}/metrics"

def training_run(
	pfl_model,
	dataset_name,	
	cohort_size,
	local_iterations, 
	local_batch_size,
	learning_rate, 
	samples_per_user,
	central_iterations,
	sampling_probability,
	epsilon,
	delta,
	clipping_threshold,
	cnt_iteration, 
):
	if dataset_name == "mnist":
		train_data_federated, val_data_federated, val_data_central = load_and_preprocess_mnist_federated(samples_per_user=samples_per_user, normalization=True, scaling=True)
	elif dataset_name == "fashion_mnist":
		train_data_federated, val_data_federated, val_data_central = load_and_preprocess_fashion_mnist_federated(samples_per_user=samples_per_user, normalization=True, scaling=True)
	else:
		raise ValueError(f"Unsupported dataset: {dataset_name}")
	
	
	pld_accountant = PLDPrivacyAccountant(
		num_compositions=central_iterations,
		sampling_probability=sampling_probability,
		mechanism='gaussian',
		epsilon=epsilon,
		delta=delta
	)
	

	pld_local_gaussian_noise_mechanism = GaussianMechanism.from_privacy_accountant(
		accountant=pld_accountant, 
		clipping_bound=clipping_threshold
	) # By default is local mechanism

	postprocessors = [pld_local_gaussian_noise_mechanism]


	pfl_simulated_backend = SimulatedBackend(
		training_data=train_data_federated,
		val_data=val_data_federated,
    	postprocessors=postprocessors
    )
	model_train_params = NNTrainHyperParams(
    local_learning_rate=learning_rate,
    local_num_epochs=local_iterations,
    local_batch_size=local_batch_size,
	local_max_grad_norm=clipping_threshold
	)

	# Do full-batch evaluation to run faster.
	model_eval_params = NNEvalHyperParams(local_batch_size=None)

	evaluation_frequency = 4
	algorithm_params = NNAlgorithmParams(
		central_num_iterations=central_iterations,
		evaluation_frequency=evaluation_frequency,
		train_cohort_size=cohort_size,
		val_cohort_size=100)

	central_data = Dataset(raw_data=[val_data_central.data, val_data_central.targets])
	pfl_callbacks = [CentralEvaluationCallback(central_data, model_eval_params, evaluation_frequency), AggregateMetricsToDisk(output_path=f"{base_path_metrics}_{cnt_iteration}.csv")]

	algorithm = FederatedAveraging()

	pfl_model = algorithm.run(
		backend=pfl_simulated_backend,
		model=pfl_model,
		algorithm_params=algorithm_params,
		model_train_params=model_train_params,
		model_eval_params=model_eval_params,
		callbacks=pfl_callbacks,
		send_metrics_to_platform=True
	)

### Run training
Note that the `cnt_iteration` variable is used to save the metrics file and json file with different names for each run, and can be used to run different experiment in sequence or a hyperparameter search.

In [ ]:
pfl_model = get_pfl_model()


cnt_iteration = 0

training_run(
	pfl_model=pfl_model,
	dataset_name=dataset_name,
	cohort_size=cohort_size,
	local_iterations=local_iterations,
	local_batch_size=local_batch_size,
	learning_rate=learning_rate,
	samples_per_user=samples_per_user,
	central_iterations=central_iterations,
	sampling_probability=sampling_probability,
	epsilon=eps,
	delta=delta,
	clipping_threshold=clip_bound,	
	cnt_iteration=cnt_iteration,
)

# Extract the max central val accuracy and save to a json with hyperparameters, and val loss
save_best_stats_per_iteration(
	iteration=cnt_iteration,
	hyperparameters={
		"local_iterations": local_iterations,
		"learning_rate": learning_rate,
		"clipping_threshold": clip_bound,
		"samples_per_user": samples_per_user,
		"cohort_size": cohort_size,
		"sampling_probability": sampling_probability,
		"epsilon": eps,
		"delta": delta,
		"central_iterations": central_iterations,
		"population_size": population_size,
		"local_batch_size": local_batch_size,
		"momentum": momentum,
		"weight_decay": weight_decay,
	},
    dataset_name=dataset_name,
    base_folder_metrics=base_folder_metrics,
	base_path_metrics=base_path_metrics
    
)